In [0]:
# ============================================================
# NOTEBOOK 03 — Gold Aggregations
# Medallion Architecture: Silver Parquet → Gold Summary Tables
# ============================================================
# Reads Silver Parquet and produces business-level aggregations
# that will feed directly into the Streamlit dashboard.
# ============================================================

from pyspark.sql import functions as F

SILVER_PATH = "/Volumes/workspace/default/flight_raw_data/silver"
GOLD_PATH = "/Volumes/workspace/default/flight_raw_data/gold"

print("Silver input path:", SILVER_PATH)
print("Gold output path:", GOLD_PATH)

Silver input path: /Volumes/workspace/default/flight_raw_data/silver
Gold output path: /Volumes/workspace/default/flight_raw_data/gold


In [0]:
# ============================================================
# Step 1 — Read Silver Parquet
# ============================================================

df_silver = spark.read.parquet(f"{SILVER_PATH}/flights")

print("Silver row count:", df_silver.count())
print("Silver columns:", df_silver.columns)

Silver row count: 5714008
Silver columns: ['YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'AIRLINE_NAME', 'FLIGHT_NUMBER', 'TAIL_NUMBER', 'ORIGIN_AIRPORT', 'ORIGIN_AIRPORT_NAME', 'ORIGIN_CITY', 'ORIGIN_STATE', 'DESTINATION_AIRPORT', 'DEST_AIRPORT_NAME', 'DEST_CITY', 'DEST_STATE', 'SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'DEPARTURE_DELAY', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME', 'ARRIVAL_DELAY', 'SCHEDULED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'TAXI_OUT', 'TAXI_IN', 'DIVERTED', 'CANCELLED', 'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY', 'IS_DELAYED', 'DELAY_CATEGORY']


In [0]:
# ============================================================
# Step 2 — Gold Table 1: Delay statistics per airline
# ============================================================
# For each airline we calculate:
# - total number of flights
# - average arrival delay in minutes
# - delay rate: percentage of flights that were delayed
# - average delay broken down by delay type
# ============================================================

gold_airline = df_silver.groupBy("AIRLINE", "AIRLINE_NAME").agg(
    F.count("*").alias("TOTAL_FLIGHTS"),
    F.round(F.avg("ARRIVAL_DELAY"), 2).alias("AVG_ARRIVAL_DELAY"),
    F.round(F.avg("DEPARTURE_DELAY"), 2).alias("AVG_DEPARTURE_DELAY"),
    F.round(F.avg("IS_DELAYED") * 100, 2).alias("DELAY_RATE_PCT"),
    F.round(F.avg("AIRLINE_DELAY"), 2).alias("AVG_AIRLINE_DELAY"),
    F.round(F.avg("WEATHER_DELAY"), 2).alias("AVG_WEATHER_DELAY"),
    F.round(F.avg("AIR_SYSTEM_DELAY"), 2).alias("AVG_AIR_SYSTEM_DELAY"),
    F.round(F.avg("LATE_AIRCRAFT_DELAY"), 2).alias("AVG_LATE_AIRCRAFT_DELAY")
).orderBy("DELAY_RATE_PCT", ascending=False)

print("=== Gold Table 1: Airline Delays ===")
gold_airline.show(14, truncate=False)

=== Gold Table 1: Airline Delays ===
+-------+----------------------------+-------------+-----------------+-------------------+--------------+-----------------+-----------------+--------------------+-----------------------+
|AIRLINE|AIRLINE_NAME                |TOTAL_FLIGHTS|AVG_ARRIVAL_DELAY|AVG_DEPARTURE_DELAY|DELAY_RATE_PCT|AVG_AIRLINE_DELAY|AVG_WEATHER_DELAY|AVG_AIR_SYSTEM_DELAY|AVG_LATE_AIRCRAFT_DELAY|
+-------+----------------------------+-------------+-----------------+-------------------+--------------+-----------------+-----------------+--------------------+-----------------------+
|NK     |Spirit Air Lines            |115193       |14.47            |15.88              |28.79         |13.77            |1.29             |27.51               |20.49                  |
|F9     |Frontier Airlines Inc.      |90090        |12.5             |13.3               |25.36         |14.72            |0.92             |24.66               |26.9                   |
|B6     |JetBlue Airways    

In [0]:
# ============================================================
# Step 3 — Gold Table 2: Delay statistics per route
# ============================================================
# A route is an origin-destination pair e.g. LAX → JFK.
# We calculate average delay and delay rate per route,
# then take the top 50 most delayed routes by delay rate.
# Minimum 1000 flights filter removes low-volume routes
# that would skew results with small sample sizes.
# ============================================================

gold_routes = df_silver.groupBy(
    "ORIGIN_AIRPORT", "ORIGIN_CITY",
    "DESTINATION_AIRPORT", "DEST_CITY"
).agg(
    F.count("*").alias("TOTAL_FLIGHTS"),
    F.round(F.avg("ARRIVAL_DELAY"), 2).alias("AVG_ARRIVAL_DELAY"),
    F.round(F.avg("IS_DELAYED") * 100, 2).alias("DELAY_RATE_PCT")
).filter(F.col("TOTAL_FLIGHTS") >= 1000) \
 .orderBy("DELAY_RATE_PCT", ascending=False)

print("=== Gold Table 2: Top 20 Most Delayed Routes ===")
gold_routes.show(20, truncate=False)

=== Gold Table 2: Top 20 Most Delayed Routes ===
+--------------+-----------------+-------------------+--------------+-------------+-----------------+--------------+
|ORIGIN_AIRPORT|ORIGIN_CITY      |DESTINATION_AIRPORT|DEST_CITY     |TOTAL_FLIGHTS|AVG_ARRIVAL_DELAY|DELAY_RATE_PCT|
+--------------+-----------------+-------------------+--------------+-------------+-----------------+--------------+
|LGA           |New York         |CMH                |Columbus      |1316         |12.38            |32.67         |
|RDU           |Raleigh          |LGA                |New York      |1677         |17.2             |32.56         |
|RIC           |Richmond         |LGA                |New York      |1283         |17.89            |31.96         |
|IAH           |Houston          |LAX                |Los Angeles   |4388         |16.74            |31.52         |
|ORD           |Chicago          |DEN                |Denver        |5517         |18.14            |31.41         |
|DEN           

In [0]:
# ============================================================
# Step 4 — Gold Table 3: Monthly delay trend
# ============================================================
# Shows how average delay varies across the 12 months of 2015.
# This reveals seasonal patterns — summer and holiday months
# typically have higher delays due to increased traffic and
# weather events like thunderstorms.
# ============================================================

gold_monthly = df_silver.groupBy("MONTH").agg(
    F.count("*").alias("TOTAL_FLIGHTS"),
    F.round(F.avg("ARRIVAL_DELAY"), 2).alias("AVG_ARRIVAL_DELAY"),
    F.round(F.avg("DEPARTURE_DELAY"), 2).alias("AVG_DEPARTURE_DELAY"),
    F.round(F.avg("IS_DELAYED") * 100, 2).alias("DELAY_RATE_PCT")
).orderBy("MONTH")

print("=== Gold Table 3: Monthly Delay Trend ===")
gold_monthly.show(12)

=== Gold Table 3: Monthly Delay Trend ===
+-----+-------------+-----------------+-------------------+--------------+
|MONTH|TOTAL_FLIGHTS|AVG_ARRIVAL_DELAY|AVG_DEPARTURE_DELAY|DELAY_RATE_PCT|
+-----+-------------+-----------------+-------------------+--------------+
|    1|       457013|             5.81|               9.69|         20.22|
|    2|       407663|             8.32|              11.78|         22.52|
|    3|       492138|             4.92|               9.58|         18.64|
|    4|       479251|             3.16|               7.63|         16.44|
|    5|       489641|             4.49|               9.36|         17.64|
|    6|       492847|              9.6|              13.87|         22.73|
|    7|       514384|             6.43|              11.35|         20.21|
|    8|       503956|             4.61|               9.87|         18.01|
|    9|       462153|            -0.77|                4.8|         12.45|
|   10|       482878|            -0.78|               4.94

In [0]:
# ============================================================
# Step 5 — Gold Table 4: Delay by departure hour
# ============================================================
# SCHEDULED_DEPARTURE is stored as an integer like 1435 meaning
# 14:35. We extract the hour by integer-dividing by 100.
# This shows how delays compound throughout the day — early
# morning flights are on time, late evening flights carry
# accumulated delays from earlier in the day.
# ============================================================

gold_hourly = df_silver.withColumn(
    "DEPARTURE_HOUR",
    (F.col("SCHEDULED_DEPARTURE") / 100).cast("integer")
).groupBy("DEPARTURE_HOUR").agg(
    F.count("*").alias("TOTAL_FLIGHTS"),
    F.round(F.avg("ARRIVAL_DELAY"), 2).alias("AVG_ARRIVAL_DELAY"),
    F.round(F.avg("IS_DELAYED") * 100, 2).alias("DELAY_RATE_PCT")
).orderBy("DEPARTURE_HOUR")

print("=== Gold Table 4: Delay by Departure Hour ===")
gold_hourly.show(24)

=== Gold Table 4: Delay by Departure Hour ===
+--------------+-------------+-----------------+--------------+
|DEPARTURE_HOUR|TOTAL_FLIGHTS|AVG_ARRIVAL_DELAY|DELAY_RATE_PCT|
+--------------+-------------+-----------------+--------------+
|             0|        14504|             0.81|         15.18|
|             1|         5091|             3.89|         18.01|
|             2|         1383|             1.82|         16.49|
|             3|          767|             1.73|         17.08|
|             4|          526|              3.7|         19.01|
|             5|       115647|            -3.78|           6.9|
|             6|       398508|            -2.61|          8.58|
|             7|       387719|             -1.5|         10.66|
|             8|       374828|            -0.19|         12.29|
|             9|       346299|             0.97|         13.58|
|            10|       366008|             2.17|         14.77|
|            11|       352601|             2.88|         1

In [0]:
# ============================================================
# Step 6 — Save all four Gold tables as Parquet
# ============================================================

print("Saving Gold Table 1: Airline Delays...")
gold_airline.write.mode("overwrite").parquet(f"{GOLD_PATH}/airline_delays")

print("Saving Gold Table 2: Route Delays...")
gold_routes.write.mode("overwrite").parquet(f"{GOLD_PATH}/route_delays")

print("Saving Gold Table 3: Monthly Trend...")
gold_monthly.write.mode("overwrite").parquet(f"{GOLD_PATH}/monthly_trend")

print("Saving Gold Table 4: Hourly Delays...")
gold_hourly.write.mode("overwrite").parquet(f"{GOLD_PATH}/hourly_delays")

print("All Gold tables saved successfully.")

Saving Gold Table 1: Airline Delays...
Saving Gold Table 2: Route Delays...
Saving Gold Table 3: Monthly Trend...
Saving Gold Table 4: Hourly Delays...
All Gold tables saved successfully.
